# Statistical testing: did the market catch on to Moneyball?

The core Moneyball insight was that on-base percentage (OBP) predicted
runs scored better than the batting average and slugging stats teams
were actually paying for at the time, an inefficiency the A's exploited
in 2002. Once the book came out and every team started valuing OBP the
same way, that edge should shrink. This dataset runs a full decade past
2002, letting that prediction actually get tested.

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats

baseball = pd.read_csv('baseball.csv')
baseball['RD'] = baseball.RS - baseball.RA

### Is run difference really as strong a predictor of wins as it looks?

In [2]:
r_rd, p_rd = stats.pearsonr(baseball.RD, baseball.W)
print(f'Pearson r = {r_rd:.3f}, p = {p_rd:.2e}, n = {len(baseball)}')

Pearson r = 0.938, p = 0.00e+00, n = 1232


Yes, confirmed across all 51 years of data, r = 0.94, about as strong a relationship as sports statistics gets.

### Has the OBP premium over SLG shrunk since Moneyball went public?

In [3]:
def obp_slg_coefficients(df):
    X = sm.add_constant(df[['OBP', 'SLG']])
    model = sm.OLS(df.RS, X).fit()
    return model.params['OBP'], model.params['SLG'], model.rsquared

pre_obp, pre_slg, pre_r2 = obp_slg_coefficients(baseball[baseball.Year < 2002])
post_obp, post_slg, post_r2 = obp_slg_coefficients(baseball[baseball.Year >= 2002])

print(f'pre-2002:  OBP coef = {pre_obp:.0f}, SLG coef = {pre_slg:.0f}, ratio = {pre_obp/pre_slg:.2f}, R2 = {pre_r2:.3f}')
print(f'post-2002: OBP coef = {post_obp:.0f}, SLG coef = {post_slg:.0f}, ratio = {post_obp/post_slg:.2f}, R2 = {post_r2:.3f}')

pre-2002:  OBP coef = 2738, SLG coef = 1585, ratio = 1.73, R2 = 0.930
post-2002: OBP coef = 2553, SLG coef = 1818, ratio = 1.40, R2 = 0.912


The OBP-to-SLG coefficient ratio drops from 1.73 pre-2002 to 1.40
post-2002, OBP is still worth more than SLG per unit, teams didn't
overcorrect, but its relative premium shrank by about a fifth once the
strategy became public knowledge. That's a real, checkable piece of
sports-economics history: a market inefficiency getting partly (not
fully) arbitraged away once it became widely known, exactly what
efficient-market thinking would predict, something only visible once
data past 2002 is in view.

In [4]:
import json, os
os.makedirs('outputs', exist_ok = True)
with open('outputs/statistical_tests.json', 'w') as f:
    json.dump({
        'rd_wins_r': float(r_rd),
        'pre_2002_obp_coef': float(pre_obp), 'pre_2002_slg_coef': float(pre_slg),
        'post_2002_obp_coef': float(post_obp), 'post_2002_slg_coef': float(post_slg),
        'pre_2002_ratio': float(pre_obp / pre_slg), 'post_2002_ratio': float(post_obp / post_slg),
    }, f, indent = 2)